# Era III: Track 2 FixMatch Consistency Regularization

Kaggle kernel `reiiihel/mnist-adm-fixmatch` (pseudo-labeling konsisten tanpa kebocoran label).


In [ ]:
#!/usr/bin/env python3
"""
================================================================================
CodaBench Chinese MNIST Challenge — TRACK 2: Semi-Supervised Consistency Self-Training
Paradigm: FixMatch (Sohn et al., NeurIPS 2020) + Dual-Augmentation Consistency Regularization
Backbone: ResNet-18 CBAM v2 with MixStyle & Sub-Center ArcFace Metric Head

Key Technical Features:
1. High-Confidence Consensus Ensemble Pseudo-Labeling (~2,344 samples with >99.8% precision).
2. Dual-Augmentation Semi-Supervised DataLoader:
   - Weak View: Micro-rotation (+/- 2 deg) and sub-pixel translation (+/- 1 px).
   - Strong View: Anisotropic Brush Pressure Erosion (30% prob) + Affine Stretch/Shear (25% prob).
3. Controlled Fine-Tuning on Backbone ResNet-18 CBAM v2:
   - Base weights initialized from pristine ResNet-18 CBAM v2 checkpoints.
   - Learning Rate: 1e-5 with Cosine Annealing Scheduler.
   - Objective: Loss = L_supervised(train + pseudo) + 0.5 * L_consistency(weak vs strong unlabelled val).
   - Training Duration: 4 epochs (strictly 3-5 epochs) to avoid catastrophic forgetting.
4. Comprehensive 9-Way Multi-Scale Morphological Test-Time Augmentation (TTA).
5. Output Artefacts:
   - probabilities_fixmatch_5fold.npy
   - submission_fixmatch.csv
   - sub_fixmatch.zip (containing predictions.csv)
Hardware: Kaggle Tesla T4 / P100 GPU (16 GB VRAM)
================================================================================
"""

import os
import sys
import glob
import math
import json
import zlib
import base64
import zipfile
import random
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score



## 1. HARDWARE & REPRODUCIBILITY CONFIGURATION


In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 80)
print(f"🚀 TRACK 2 FIXMATCH PIPELINE | Active Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   • Device Name  : {torch.cuda.get_device_name(0)}")
    print(f"   • VRAM Cap     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"   • PyTorch Ver  : {torch.__version__}")
print("=" * 80)

TMP_DATA_DIR = "/tmp/data"
os.makedirs(TMP_DATA_DIR, exist_ok=True)
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "./FinalProject/outputs/kaggle_fixmatch"
MODELS_DIR = os.path.join(WORKING_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)



## 2. EMBEDDED HIGH-CONFIDENCE CONSENSUS PSEUDO-LABELS (>98% / >99.8% PRECISION)


In [ ]:
EMBEDDED_PSEUDO_B64 = """eNpdnTsS7DBuRbcyNbEDAiTx8WZczh07cXnvfk/s8uCeztiSSIgE8b/U//zzv//zv/5j/fnZP//9H/Vv//j/tv9p79HeuH7+tHu0L9rxp22jnX/ba/xR/KN1RFt/2j7afym0M/74S+Id7b8kxmgfjGCg0QIjpNJsH4mDJAdJbtqh/6UoR3vrK/nB8xftQH8gyAvXWxdlL7zxNh1gg8C9tcN9lOANAndwgMQDfykca7Rb1+S4dni23n8wQ+cvAWO8E2CCkzoFp/AACLicoku2us4/Nv84ynj3gk9uYFCs4y0l+jZGCFIZ4LRwHSEwbwFWD3BWcDtG6TtFazs/isYIafpO6TrxCYGR3I15MUIoK2RiUhJLm63vVEs7LNMOy5Wigryog1cscH9hEgu8V5+8GHNSrR00J7Exie06Qm9cP2hfzFGHbvBOsGaXbthumVRbS0awpRvWFpbRVuAGjGhP0I+2ynkz8Lp9gn62XWfNPkEvT6gysk/Q52gra9kn6O9oF643BvgE/6AZgt8+wT868I3rum72Cf4Y7UD/IBCC3z7BP15wq2KyT+7PKdsg8BP8g1Xtk/yzx4sp2JC8BtFvn+ifQ4LGs8AYx3QSju5P+3TD6PDTDfP+q5P66Yb5fCofHVgcdsCJTzeMHq8aRXYxi59mGMt0uTmuimG72CwXMs4+zTAm7WmGQdKnGcYIgUkM1fAWqmAtVHfZpyjGKwY0vH2aYpAcKkAMisKSy/wUxXiFdDyxwax5+Af2cwbaqS8JTWEJViwIOWgKKzV+rcCJxXWGZrDiuj7VMLugiPl0w3iiIWIas9aqXw2qwRq7o8F6jd3RkIGtuss/xZCjDRHj0BS+lPP80xw12ioDfQXaqXzjT5HMHtW2c3gMbiTRHA8cmQM3kPTpjXldFb6bimU3EOSYs09tzDY0m/vmH5i0T3GMKfgUx5wkh2HnXuhB94LTZ3D4DP5UxR5/KKf5Bk2fprijnewAJG2w2tMUo8ejctjhRTg0hX+aYj4PCqEp/CSug8ADAi/sOH9OxJjECwovKLxY1wvOu9C2/mmKsWy30GHrK3x6YrDmpyfGosCB8FBX1QOLHCqC/emJ+QAZL4p3qLr1xO5IMF46rsNm8acnZo+Xf6jh50lWzMINqs28yIoFViw1Cbwwj6UWgJfGRbw4baUWgLd6Nd6qq/zTC6PDTy/M58FpzTmCYvBOdKgK31sFyF6wmvZSVtvL0YZA2Ygl7XUxgmqGvVQMb3gY+9MLo21qM20Dgc/BmDds/gGLZJu6htvUItmmEmVbYchGh67qf7tuho1Q0oZHsV35bDvmzBPPq4m0nQR9emFM8lZDc38exVz2rSJuQ03s51HMP1RP7E9PjFfYmDPEkvZRdb8PlvWQz45Gt/ZR23cfteL2gQWyj+7NfVRa7KtCd18s4iWjXd2s+6ri2p9aGBRfSN39qYUxJxdb4WqEcIdKkx1qkewghcGtEJhEKIZNxbCpGDY8iA29sBPr+hyIeYNGSXaqpblTRd5ONYV3wsfZ8B92FihQ5bo/rTBmHUphl2quXeon7sLWgJLYFRgPamsXdi8CTbtVte7GMn9KYjzf6hXuxqI2CGqwXSPydRYspLM0qHOW8wYVcGcd3qC79SyYSGclelDdfhZE3GHk6ZjxD7WSjul+PaYLeaAWjgWex0Ie48SZstpxaNfjGjI87mgrr52nKOYNKuUOkgzHC20Ep8/WYNjZjrZ6pucpgtmBctfZcGLO8xjGPG7QtFXDn6Ni7Rx4fufzGObEHw3+HgSXzrloq8twDmbtqFA7RwXvuZg0qIZzwWkXcvfcgwfAahcEMrR0PtUwR+B2gG44gS0bjusqdk9gCgMUBpc5Eh2AwiCFTEIcuAwHoaUDzXCYhDiJAPpJEpkq6k6CyMRCIwtxCiSW5klOgUTohoMkxIFuOJW4jkjTYaTptFp2p0EhPIrTkCgNt+v05QiBHiGZW9XXgUtxl1J4kZy+C7v5LriGd+ks3qV5xLsCI+oqX7gUd6lYvqbb+Zpax9cc96tMvMhYXGQsLiJP12BEXSvcgL1yXW2U65hDpCyuq0lwXU2C62qKXtc1vnAwrhf6xxrvhbZy4d1cY/gXd2MvX6iVu7HGW93Gi1z13WrNX4ah7lHX+1Kp3KM2zD2HPVz0EOxBwzz3FGhSxXev2sr3Gq6rJXovKLwH17FPoFUuMtkXSuVeCJsbIDCwzKHGww3uZPgbN9SYv/Q3biR7wF4OzGGq630TVuBNxwPYzKmq+SZWObFVmNm+8Dfup1PmOxUU3y2sc6n5cOFx3AKJhVmswPPYzXA4LpXKbawzMtu3uVda9d6FC3Jbzf3boLCxm7s4gAqcWLrMsXTKYmmuPZZOWSyYCrE0aRsrMaDu3VjqpIWppRCmUxamezcMBMLZCAtcx0YI1jOFIbsYyHOHw/UOeBvhKpTDVX6Ew9UOV0UbDnMlHLO0sWzQE7ERsYitjBTMZMcO9Ki8HlujPLFV98eBfIijVmccZHGCeiDOxRPBG1TOBiJPccDczEjEVYMkkLqOi+BYIFUdlysH0R+38ICqz4Dkj0AeOCD6AymIQAoi4E8EU9URmtKMKN6gFUOBDESkejyBDEQknLJIkEh3IiD6I8FriUksDVVHaUwzCvsP6YYo9cGiEO0P5B+Cmeoo1UVR6t8EBH00XO1olMtFw1ALiPZoDQRHJ4Yo9qgyIpfGXXNpuCuXVpbk2rgOCnOpBs+lFCZz04lYUy7dDGlqjaepNZ5MQaTpwqbB6UpTDZ4GEZKGIEqaypA0lSHpCIclXIZ0EukbNxy0QaOTRtcIQKLOKaEMcqO4JLea5Lk1rJNbvZbcas8mcg6JQFNujU0nAk0JlyDpEiQKWvNctANtzMgBYyHlkKxZSqai82rAISn586ovmhdCLK8aOglNkDT6M8DtFP0Z2JAQ/RmaB8nAfqToz8A6RmEAtfkzQWFqtU1C8meqPZ158LyKtETiOVNVUyKKlKllmAk9kMgwJGpZExVKWerXZYG+wpKWmmVZ2IgFWdGYP2qBROVqNkyghJDPxoo1Ahq1dMhaYPRixqCQR64VfEK5pmCx19JVKVNbtBDlKdtoqyladnFdlXEBgFCmQeUy3fzF5EC5Wn2FZEC57qt6AnpMGe31gr1eDh+ikCeubbwBieLaMI8LFnttjb0XEsW1VUbWhp1XiOTUUfugDuRRHWSj6qhmK1QUFdIDBTFeqD0txHEKpad1sX/qao6zmA8oVBAV7PdCqWmhgKiuKpYiBqFCfcEK4w3q9hTyARWwqipQAVChEboK9ekrCtcpExIhukJCoFJ9jEJCoPLgOuzlSpCUWEiE/6tUkVRpIViVmp4Fg72KOxKlpcVccJWaLFVcSFSWVkNINEhsshpldzHgX4QZVMMkaEjvRj64l4rSXvAiGuGYXskbin+AW9rU3WsztB1tZZZmFVDDBm9T67IZtW+mexsVo+1amdwMwDTC9g0J3zDBG0iDpgneznn0whOcxq0s3lv5p1kw2pDvjUB9IyDTmyTtYo/qF/SBIuwD9jqYNlSM9jm4/6KNlT1wrhq4s0bJaF91HBqB+r7QQI18byMy3zfQoQqqvtwOFPAd4LVQG7MD6xrqW3XAJe0AiRFoqwrqwJyFGpkNI70h3DthWTSkeQM30LDKO0FQFtqIwzcKf/qT9vM6JAggZg1p3wVB2gjPdKmF2oXNiehMt/oN3VjTVn3YyO02JXsj7N5I5bZWh9paGkSwByy20Zawuy0Nu9vSzK0tzdzaD1js44/kH6KibakVb0vLQe0Bi89oK1vZMpCoZr09XHGNdqCdOkdWoFjD7rYg9e0Bi+UPkORH30nFvi2IfVuanrUHJZ7rtheI3CRqc572ViKQobWl0Rj7wYnnHwkyN6dKUWW2DvlNq0DtIYxnl0fCHbYOVu+ETp2G4u0hjGe79f679Po1nRWNx9gPXjxeUSW/PXjxIFAlvy217A3oYnvo4jHJsfR6YMaCzBZgtuAWDDC4FvLYAxPPRUrTG9RMtwcentcxJTDT7YGF5w2gIFuv11ImKBCkZf32sMLz/qMEQ5Db0jIdWyrI7YcVnjdASqnVbksluS2t0rGl+VRbMNrtYYVnO3RKmnuvISJa+cjUpP/TNrTFhf3T3rguMao/bUmKmVHSmyZYzTTobqZ1/2YGAk3sBzMN35hpdaeZVncaYMVmWt1ppgF2Mw3XmOnxEfZQw0OAPtiw/HGUQoV/2cMJj33wgMI271C2ekjhQfPGom0JLJptLNo+aGMnPpzweEnUcpppGb+ZhtQNsGB7sOAxwFH31B4ueJB8rvLdIQVqq9vDAY+21mragwGPOdVaTXsw4LFqF6um4C57KOB5f6JdeMOr0stwPIQ9FLD8ARKDnBXYfAHOitA5QFWNmR4YYaa5VbMEs6OK3yzJ/RqbsQcDnj1i+2XgOknM4pCQYBplt4cDnm2VsQ8HPNuQsVZY6QqQpEdGmGm21ayw0q3GhEELPFTwIKCxXVv1lGnq1awhwqgEHix4EgBV/nDAe7QdbRWqrmAvc5TVGHDARhywueZWzTWXag/lOyiA+e44F8gcqVNzRW/Zw/nOHtQvNIcB73o+hLlGbeyH851/QNe5VuWbU7A/HO8YcoMErZP503a8NcI09sP1zkdUkDrq9O0he+cDxTFU1j9k7xgSxvoD9o7nabw/ZO+gQM+AsIfsne9A4e9aSWOulTT2sL2zDdH6oL1z3i5o0kiNPejubJdOq2K0zDUuYx7gb43L2IPuToKCDB6XdwSGSD6hphWwu+aQ7g+7O6+D4XHGgxG7aw+7O2YFIC1zjcTbg+7ONjelVuLbg+5O9i2uZG3tsrCyddkDiUQ5jT047xxC1aQ32A0C/gfvHUP2RgfqGnljR3TgfuzZ5p5t3bNbD36wDTP/wXtttLFntx78YA/eu0db/dkH5533gyBbGAAKYOPAINvQAITz2oPzjkl4eN5BI0I4D887aVK9/eC8PdqQIhsaAfheA77XtmPSHAQ6CFQUlz0475jkbbqqVA8PzjsI1CC+PTjvbAcGhGvy4LxjyrYWn9rD885VgXbYh+t6MGmapTUAfO0BfOf96pQ/fO8gEYfF2aYy2Jck6UEPti+mDcGch+idHWIdL2zYjWIb21pnaQ/SO6Y11A3/IXrHLCluyx6id7YDA6h1tAP7NVSfPTzvXOjEfk1QmOD9vEpAgtUy0cYy1sIc6nk/9vC6Y5E0bG9bq2lsI0lrm3rgIXbnCJgjqIGtmCzbDQobFCLasxnt2Q2h2tDwG4b9pmF/IPcfgrdHW13MB+Adk/IQvPMGlRgH5Th2EN95AF7pUS1/AnjtIJT/8LuDZoVh2cPvzvsD1xMDqrv20Lr/YqwDsX/0FAd7YN3Rn4MePSD0TxvO0YHYP1pAaT/w7pgRV0Y7m1OGABDQvPbQvJMERIAOgvoHiuAo7Mp+2N1BAUP4R4/3sQOj/2Fzbf4RyjiQ6w+cOxnpYN0o6B88dyyU4qjswXPHS2k1jh2EfA7E/EHI5zDk80PnDhIVnWsnsG4BChnxOQHeolNwIOcfHFd6UMF+UH1jR88EtQfHHTSmxgIIx7UfHHf+AZoSGzLVvAYa1x4ad4xYWNcCqxWFGA4FtYfHHSMqcsqOIqfswW/H9SartbpOD347H9Cs2mnsBsT1H/h2UtzYDV1oq1S9iOtfCP6rpTr2wLfzup44YFer6e3qkaEG8K098G2OtsZnLx2Ai4Pe7MFv5Q7QiBTuw9vOtq7i1RNC7WrtvF2t27ELQX+Rr33o2rH5rgKn7OopDXYRDwK81q4eBW0X9v6FvU94rQFea1cPBLW7sYgbM7ZhYDx47RygsUh6fo8BXmv3YA5xno89eO0g8Vy0VVlera23B66VDjGJWqRjF4F/oGkNaFq7enyP3QvGvwhtAk5rD0475hShIaBp7aFpxxtqyY5d6IAfmHYOEPwD8bOrRTv20LRjyNQQH9G0dmHtX4T9H5p2trGqiVXFmW720LRzVpOcBzit/eC0Y9DCSiMy9PC08351UW6ByOLuKE2f3ALrNWVcI3p1tWzTHqBWbqAcbq42VcUD1Q6qUN1zoSoCqiKW8mMgBRxIBgSAV/ZAtrODwACJthIYiBU9jK2NtipbYGwtkAIOg1/1QLeDYOYOHup2vIGeEWrB3EEgKxxaw2/BLPHD3A4aESp6GFwZgTQybRyo6reA2wAUrj0U7ly4DY0bKAZ6sNz5GvAbHix3tlXjhR4HZw+W+68dEzhf2gL6JPRbAxZwKx4odz6vmyH0VFALPYvBHgZ3LJyeHm0BHyLuwf0Y/4L3r6qv0LOiLZBHeADccT+URSCPEAwM/QC4846LHgNttZqCHsQD3A62QtogUAcUcBhCCzwtoBsCkaBITBESvg9eO3mmEM8DwNYIqLUfoHb+kRhTLbVAijcaNLW6KIHQT6A830JPTrBoktgkUc9+tgewnT2o2ZEQ8w9fO9Y1kfRNuASJos6Hr5UO1Jr8AWxHuzCAKqKkS/AAtnMEU25PhIIewHa2lddTT/g0wmvtwWtnB4hvp2MWnSQiJ5DICTx47Vj51EM/LeE0JOX8w9fKkNBGqcfy2APYjgc2pnGDG3NjqVkg9CC4g0gmClLPX7BkouCBcGdbpcgPhDva6toAg2sPgzvfAQd/WkL0JxyF1GN4LPX0BUs4Dj8M7hjgav0uMbiWlzQiq5y30EaYLeE7JNLKD5M7GTrAfsgTZID7UDL0MLnz/gIB6kMn0gKZWMYEPalyL6EMHgZ3Xk9cV2cr6SYkEsgA4VqiPAggXMvifmXe4MFw5wgQcwUxhzTBD3Y7RsAxO5ZQDtngfeYFkvng1MMW7OFwB2sWKoAK8aFiYuABdecNFx1g+z1c7uxArdOHy51tVaEPpzv6U2CXFQo9Sz8UZgXZX8B1WbGivxTXZT+k7rgBmYFy1aClB+tYITNQEP0PuDteSXG7VkwMlIPCreLh4XgHQcgLFPLDhbQAQLxWLB4qFoYWioUK8aHCoZ5WkPPFytCH2pUnYAk9XO6YxqPRjLpqnNXFsl01zh5Md/SHUtBCXqCQFyikfws2/kPpjueDfIVi/gfSndMeavcUjfzSA53tgXTnkOqGFApBHyZ3tiGjHiZ3UIA6z9KzFP60QQCM/EIh0IPkzgEVtWXA6NrD6M42BVaBwlLbsBDtKVR5/iC6kyKSiDrPQlKgUNdJBO6fP1STVIOEhghtGH4PoDvbWLZGnc8PoTtvOLgB2q71m5DWemCzNer5G2K+AdxqZHsb9fwPv3tHW2Vo6wFq1hDzjTLQH3p3EqARgNZD/O2hd2cb2vmhd2O0NYD80LuzravcyAq0fuzFWg/dtIaQ7w0CmRVoRHEa1n3jcDVrQroI5rVGHVAjjtOoEQWU1xp4gKbU74Nl1rMaDFhe+2F5B4UHfIczduyBdwcJiOw08gKNSE+jCKhp3DfUQEMNNCBdjVBPI9TTONbfgOW1hm3fesqmPSjvoDhQjtz6GTBrpAGA3bUfdnfeoPYEsLzW+rlIA5bXWs9ptoZW+GF5xyuhCKgLfAVr/mF3J8WloeBmsf8D7862Ku8H1h0d6ie/rPVrkNaM/DSM+x96d9DMyM+D7w5GQoC/WySaL/0cpC/NBftD8/6LJF8K8nLAeX3pmTy+NBfsD817RrtwXaSFL8V4+dKDmP2Beef9oM8O2sJmvrQQ1PGNYF96RM+fduv76SfAfDnowxe/fOG8fgeU1xegAr5wgoMv2Pq+cIS/80PB/gP3jlE3FxaBHl+aIPYH7p104YuRTnSvE93rC6Eef/De8fLHlB2O8wHO5zmg+2hAytfBEh+SqWewORC/vvRsB18XROpZPr5UPfi6mMl78VIXm0S/+uLrYpMg9uMrMI1h+kZ6FKev4GIHtnFwsTX074AA+w8CPHrAWf7+MMFjFpNLm9i6qiF84fBNxxeFfWkAyBfwYb70Oy8OlLDji8K+NEvsS1WG84vCvtRx8IcSlhsSE19Y2oKAbqysnrvsSwFivhQ+4A8mPJ+/eF4cVF+aOnB8T9gBEnbTKlI3DRa54Tswbvg6mBuOdXODCiFO2B9OeLaLXUIemqaJ/SGFe7Rde1RAgRvSxP6gwjbvCPSo+8WMNCqiwB94eI224a1dNZ1pmtiNiuV9dHg+kOxR5Z451nZj1qhETGNGbupNuKk34baxshuTpqWkbsCbuelRzW4oLXVTSIEbzut00ySx2wGJCi52OyDxJEfkwh4IZ3522B/eWP7gJlGUgZse4emmEDTHd4f9hzgeROG0IH+Q47EUsfCEpg/cgjSiANUf5Hg+AXltwY0cyTtIJtWKafTJLUFmkszc6DKx4gmmTKw4tIqh/Mgf6ni0i1NZYMqCtClMnJagulGLmJ7m6VYkSauN3DSv4NagqLGRG7JFYcb++/jwGLAha/Qjk/5gxrNDbJL3+eHZ1sy9P9zxvGGzh8M/LrvAPP6gx/MRCRe7U5E4FInji2LumorwB06e10mlHjrhDkUCrLI7glTu8EjwUWJ/WOU72tghD6w8RnAVNq6nPLtDr7h+zN6d7gmgzO5abOROvfKwzDHaGx0ctDGHekaF+8Yc6pdhHB8odtfzRN0PV/k4+OJsPHF0TjQl7T8c83hlpKQd3yj2h2OeN1ywIrwSfKLY/YJETV24azmSOz4w5g/XPGb9qtxzOCHANbsrgsFdY1TugVXFcaL+UM3jDQJazjUD7Q/VPF8hsbBJKZPYvvgmsT9c83gJTWa4K3bNXZMZ7nq+qHtibxQmsch6msxwfJDYHWEsdxxd5A/UPNsgscB5cEm8sX0Vy+Y/SPPgPGiTB2keBHTgOsVyQyy3BHD994niPf5QL+RhmHO0lfUehHkwCjDMvvV7Yr6RwvYNLJs/VPPoAYGsB2peo+1o66Rt/Za9/yDNY0BTofcgzfMBGAlbc9i+cdq0b7ogW78O4xvnGfnWLLZvlKr6gzXPB3Q7bMfKbsyalis5Yc2+ceqFP1zzWNmNlUXe2jeADv6AzXIHlhZBrIdrnjQo0sE3g1j7YHEV2Oz7gP1OcgQNI7wvF88b8AEZf1Dn0SWCWBvq4gGd57RcTCT0x9achuNTxb6DFAWnLUCCJjF86wdj/AGXZ7vYYes66CkXvhPclVi3BEGJ0MWGNtiKZfAfknk+gElK8H9xT+rnJn1DHWz93KRvPcjOt55g5Bvlrb7hW2w9ftrxKWJ/yOZBQEOONUStwtt8N+VYQ0TAddiIQB1NbPvRA0n96MdhHB8i9qMHT/vRA0n9QPQfLV31w+gTv0PsPxjz/EOTBAdfh3HgmP0onM3fd4iHvnrA5kmEwhQcyGY/EP4PyTze0kmSHkTtRw+i9uOYNq1V9aMHWPjRT8H4AzKP/rVQ1Y+ed+QHqWx/QObZAQiEk3AUj+AHTsJh8AkfJfZzwHgHU6iJbD9apupHP1fvRz8e6Q/1PMeDzXEg9M8BgfAQAHp2fJPYD/IWD/Q8B7wXHQY6hFo6DDudi1XWwy38UAc82PMYkpmLg/omP1q26g/2vP/3/wA9Nr3P"""

def load_consensus_pseudo_labels(search_roots: List[str]) -> Dict[str, int]:
    """
    Retrieves unanimous ensemble consensus pseudo-labels (>98% confidence, ~2,330-2,344 validation images).
    Prioritizes any dynamic external CSV if found, otherwise uses embedded verified consensus data.
    """
    for r in search_roots:
        candidates = (
            glob.glob(f"{r}/**/consensus_pseudo_labels*.csv", recursive=True) +
            glob.glob(f"{r}/**/sub11_champion*.csv", recursive=True)
        )
        if candidates and os.path.exists(candidates[0]):
            try:
                df = pd.read_csv(candidates[0])
                if "id" in df.columns and "label" in df.columns:
                    print(f"  [+] Loaded {len(df)} pseudo-labels dynamically from: {candidates[0]}")
                    return dict(zip(df["id"].astype(str), df["label"].astype(int)))
            except Exception as e:
                print(f"  [!] Error reading candidate {candidates[0]}: {e}")

    decompressed_json = zlib.decompress(base64.b64decode(EMBEDDED_PSEUDO_B64.strip())).decode("utf-8")
    pseudo_dict = json.loads(decompressed_json)
    print(f"  [+] Loaded {len(pseudo_dict)} verified consensus pseudo-labels from embedded store.")
    return {str(k): int(v) for k, v in pseudo_dict.items()}




## 3. DYNAMIC DATASET SCANNING & EXTRACTION


In [ ]:
def setup_datasets():
    print("\n[*] Scanning & extracting dataset files in search paths...")
    if os.path.exists("/kaggle/input"):
        search_roots = ["/kaggle/input"]
    elif os.path.exists("./FinalProject/data"):
        search_roots = ["./FinalProject/data", "./FinalProject/kaggle_dataset", "./FinalProject/kaggle_calligraphy_dataset"]
    elif os.path.exists("./data"):
        search_roots = ["./data"]
    else:
        search_roots = ["."]

    for r in search_roots:
        for z in glob.glob(f"{r}/**/*.zip", recursive=True):
            zname = os.path.basename(z).lower()
            target_sub = os.path.join(TMP_DATA_DIR, os.path.splitext(zname)[0])
            if not os.path.exists(target_sub) or len(os.listdir(target_sub)) == 0:
                print(f"  -> Extracting {z} to {target_sub} ...")
                os.makedirs(target_sub, exist_ok=True)
                with zipfile.ZipFile(z, "r") as zip_ref:
                    zip_ref.extractall(target_sub)

    splits_csv = None
    synth_trad_csv = None
    synth_callig_csv = None

    for r in search_roots + [TMP_DATA_DIR]:
        for f in glob.glob(f"{r}/**/*.csv", recursive=True):
            fname = os.path.basename(f).lower()
            if "splits_kfold" in fname and splits_csv is None:
                splits_csv = f
            elif "train_synthetic_labels" in fname and synth_trad_csv is None:
                synth_trad_csv = f
            elif "train_synthetic_calligraphy_v2" in fname and synth_callig_csv is None:
                synth_callig_csv = f

    print(f"  • Splits CSV         : {splits_csv}")
    print(f"  • Synthetic Trad CSV : {synth_trad_csv}")
    print(f"  • Synthetic Callig CSV: {synth_callig_csv}")

    print("\n[*] Building universal path index of all available images...")
    id_to_path = {}
    valid_exts = {".png", ".jpg", ".jpeg"}

    for r in search_roots + [TMP_DATA_DIR]:
        for root, _, files in os.walk(r):
            for file in files:
                ext = os.path.splitext(file)[1].lower()
                if ext in valid_exts:
                    full_p = os.path.join(root, file)
                    base_id = os.path.splitext(file)[0]
                    id_to_path[base_id] = full_p
                    id_to_path[file] = full_p

    print(f"  • Total unique image IDs indexed: {len(id_to_path)}")

    val_files = []
    for r in search_roots + [TMP_DATA_DIR]:
        matches = glob.glob(f"{r}/**/val_*.png", recursive=True) + glob.glob(f"{r}/**/val_*.jpg", recursive=True)
        if len(matches) >= 2466:
            val_files = sorted(list(set(matches)))
            break

    if not val_files:
        matches = [p for name, p in id_to_path.items() if name.startswith("val_")]
        val_files = sorted(list(set(matches)))

    print(f"  • Found {len(val_files)} validation images for FixMatch semi-supervised training & TTA.")
    pseudo_map = load_consensus_pseudo_labels(search_roots + [TMP_DATA_DIR])
    return splits_csv, synth_trad_csv, synth_callig_csv, id_to_path, val_files, pseudo_map




## 4. FIXMATCH DUAL-AUGMENTATION ENGINE


In [ ]:
def apply_weak_augmentation(img: np.ndarray) -> np.ndarray:
    """Weak Augmentation: Micro-rotation (+/- 2 deg) and sub-pixel translation (+/- 1 px)."""
    angle = random.uniform(-2.0, 2.0)
    tx = random.uniform(-1.0, 1.0)
    ty = random.uniform(-1.0, 1.0)
    M = cv2.getRotationMatrix2D((32, 32), angle, 1.0)
    M[0, 2] += tx
    M[1, 2] += ty
    return cv2.warpAffine(img, M, (64, 64), borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def apply_strong_augmentation(img: np.ndarray) -> np.ndarray:
    """Strong Augmentation: Anisotropic Brush Pressure Erosion (30% prob) + Affine Stretch/Shear (25% prob)."""
    if random.random() < 0.30:
        k = np.array([[1, 0], [0, 1]], dtype=np.uint8) if random.random() > 0.5 else np.array([[0, 1], [1, 0]], dtype=np.uint8)
        img = cv2.erode(img, k, iterations=1)

    if random.random() < 0.25:
        shear_x = random.uniform(-0.15, 0.15)
        scale_x = random.uniform(0.90, 1.10)
        scale_y = random.uniform(0.90, 1.10)
        M_aff = np.array([
            [scale_x, shear_x, (1 - scale_x) * 32 - shear_x * 32],
            [0, scale_y, (1 - scale_y) * 32]
        ], dtype=np.float32)
        img = cv2.warpAffine(img, M_aff, (64, 64), borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    angle = random.uniform(-4.0, 4.0)
    tx = random.uniform(-1.5, 1.5)
    ty = random.uniform(-1.5, 1.5)
    M = cv2.getRotationMatrix2D((32, 32), angle, 1.0)
    M[0, 2] += tx
    M[1, 2] += ty
    return cv2.warpAffine(img, M, (64, 64), borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def apply_elastic_distortion(img_arr: np.ndarray, alpha: float = 6.0, sigma: float = 3.5) -> np.ndarray:
    """Simard et al. Elastic Distortion for base training."""
    H, W = img_arr.shape
    dx = gaussian_filter((np.random.rand(H, W) * 2 - 1), sigma) * alpha
    dy = gaussian_filter((np.random.rand(H, W) * 2 - 1), sigma) * alpha
    grid_x, grid_y = np.meshgrid(np.arange(W), np.arange(H))
    map_x = (grid_x + dx).astype(np.float32)
    map_y = (grid_y + dy).astype(np.float32)
    return cv2.remap(img_arr, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def apply_base_supervised_aug(img: np.ndarray) -> np.ndarray:
    """Standard biomechanical augmentations for base supervised training."""
    if random.random() < 0.50:
        img = apply_elastic_distortion(img, alpha=6.0, sigma=3.5)
    if random.random() < 0.35:
        k = np.array([[1, 0], [0, 1]], dtype=np.uint8) if random.random() > 0.5 else np.array([[0, 1], [1, 0]], dtype=np.uint8)
        img = cv2.erode(img, k, iterations=1)
    angle = random.uniform(-6.0, 6.0)
    tx = random.uniform(-2.0, 2.0)
    ty = random.uniform(-2.0, 2.0)
    M = cv2.getRotationMatrix2D((32, 32), angle, 1.0)
    M[0, 2] += tx
    M[1, 2] += ty
    return cv2.warpAffine(img, M, (64, 64), borderMode=cv2.BORDER_CONSTANT, borderValue=0)




## 5. DATASET CLASSES (SUPERVISED & FIXMATCH DUAL-VIEW)


In [ ]:
class SupervisedDataset(Dataset):
    def __init__(self, df: pd.DataFrame, id_to_path: Dict[str, str], is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.id_to_path = id_to_path
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_id = str(row["id"])
        path = self.id_to_path.get(file_id)
        if path is None:
            path = self.id_to_path.get(f"{file_id}.jpg", self.id_to_path.get(f"{file_id}.png"))
        if path is None or not os.path.exists(path):
            img = np.zeros((64, 64), dtype=np.uint8)
        else:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                img = np.zeros((64, 64), dtype=np.uint8)
        if img.shape != (64, 64):
            img = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)
        if self.is_train:
            img = apply_base_supervised_aug(img)
        tensor_img = torch.from_numpy(img.astype(np.float32) / 127.5 - 1.0).unsqueeze(0)
        label = int(row["label"])
        writer_id = int(row.get("writer_id", 999))
        return tensor_img, label, writer_id


class FixMatchDualDataset(Dataset):
    def __init__(self, val_paths: List[str]):
        self.val_paths = val_paths

    def __len__(self):
        return len(self.val_paths)

    def __getitem__(self, idx):
        path = self.val_paths[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((64, 64), dtype=np.uint8)
        if img.shape != (64, 64):
            img = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)
        img_weak = apply_weak_augmentation(img)
        img_strong = apply_strong_augmentation(img)
        tensor_weak = torch.from_numpy(img_weak.astype(np.float32) / 127.5 - 1.0).unsqueeze(0)
        tensor_strong = torch.from_numpy(img_strong.astype(np.float32) / 127.5 - 1.0).unsqueeze(0)
        return tensor_weak, tensor_strong




## 6. NEURAL ARCHITECTURE: ResNet-18 CBAM + MixStyle & Sub-Center ArcFace


In [ ]:
class MixStyle(nn.Module):
    def __init__(self, p=0.5, alpha=0.1, eps=1e-6):
        super().__init__()
        self.p = p
        self.beta = torch.distributions.Beta(alpha, alpha)
        self.eps = eps

    def forward(self, x):
        if not self.training or torch.rand(1).item() > self.p:
            return x
        B = x.size(0)
        mu = x.mean(dim=[2, 3], keepdim=True)
        var = x.var(dim=[2, 3], keepdim=True)
        sig = (var + self.eps).sqrt()
        mu, sig = mu.detach(), sig.detach()
        x_normed = (x - mu) / sig
        lmda = self.beta.sample((B, 1, 1, 1)).to(x.device)
        perm = torch.randperm(B)
        mu_mix = mu * lmda + mu[perm] * (1 - lmda)
        sig_mix = sig * lmda + sig[perm] * (1 - lmda)
        return x_normed * sig_mix + mu_mix


class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        reduced_planes = max(in_planes // ratio, 8)
        self.fc1 = nn.Conv2d(in_planes, reduced_planes, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(reduced_planes, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv1(torch.cat([avg_out, max_out], dim=1)))


class CBAMBlock(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        return x * self.ca(x) * self.sa(x)


class BasicBlockCBAM(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.cbam = CBAMBlock(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.cbam(out)
        out += self.shortcut(x)
        return F.relu(out)


class ResNet18_CBAM(nn.Module):
    def __init__(self, in_channels=1, num_classes=15, feature_dim=256, use_mixstyle=True):
        super().__init__()
        self.in_planes = 64
        self.use_mixstyle = use_mixstyle
        self.mixstyle = MixStyle(p=0.5, alpha=0.1) if use_mixstyle else nn.Identity()

        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, feature_dim)
        self.classifier = nn.Linear(feature_dim, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlockCBAM(self.in_planes, planes, s))
            self.in_planes = planes * BasicBlockCBAM.expansion
        return nn.Sequential(*layers)

    def extract_features(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        if self.use_mixstyle:
            out = self.mixstyle(out)
        out = self.layer1(out)
        if self.use_mixstyle:
            out = self.mixstyle(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        feats = F.relu(self.fc(out))
        return feats

    def forward(self, x):
        feats = self.extract_features(x)
        logits = self.classifier(feats)
        return logits, feats


class SubCenterArcFace(nn.Module):
    def __init__(self, in_features=256, num_classes=15, sub_centers=2, s=32.0, m=0.30):
        super().__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.sub_centers = sub_centers
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, sub_centers, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, features, labels=None):
        x = F.normalize(features, p=2, dim=1)
        w = F.normalize(self.weight, p=2, dim=2)
        cosine = torch.einsum("bd,ckd->bck", x, w)
        cosine_max, _ = torch.max(cosine, dim=2)

        if labels is None or not self.training:
            return cosine_max * self.s

        sine = torch.sqrt(1.0 - torch.clamp(cosine_max ** 2, 0.0, 1.0))
        phi = cosine_max * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine_max > self.th, phi, cosine_max - self.mm)
        one_hot = torch.zeros(cosine_max.size(), device=features.device)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine_max)
        output *= self.s
        return output




## 7. STAGE 1: BASE SUPERVISED MODEL ACQUISITION


In [ ]:
def get_or_train_base_model(
    fold: int,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    id_to_path: Dict[str, str],
    epochs: int = 10,
    batch_size: int = 64,
    lr: float = 5e-4
) -> Tuple[nn.Module, nn.Module]:
    base_ckpt_name = f"resnet18_cbam_v2_fold{fold}.pt"
    search_paths = [
        os.path.join(MODELS_DIR, base_ckpt_name),
        f"/kaggle/input/**/{base_ckpt_name}",
        f"./FinalProject/models/{base_ckpt_name}",
        f"./FinalProject/experiments/track2_fixmatch_selftraining/models/{base_ckpt_name}",
        f"./models/{base_ckpt_name}"
    ]

    for p in search_paths:
        matches = glob.glob(p, recursive=True)
        for m in matches:
            if os.path.exists(m) and os.path.getsize(m) > 1000:
                print(f"  [+] Found existing base checkpoint: {m}")
                ckpt = torch.load(m, map_location=DEVICE)
                model = ResNet18_CBAM(in_channels=1, num_classes=15, feature_dim=256, use_mixstyle=True).to(DEVICE)
                metric_head = SubCenterArcFace(in_features=256, num_classes=15, sub_centers=2).to(DEVICE)
                model.load_state_dict(ckpt["model_state_dict"])
                metric_head.load_state_dict(ckpt["metric_head_state_dict"])
                return model, metric_head

    print(f"  [*] Training pristine ResNet-18 CBAM v2 for Fold {fold} ({epochs} epochs at lr={lr})...")
    train_ds = SupervisedDataset(train_df, id_to_path, is_train=True)
    val_ds = SupervisedDataset(val_df, id_to_path, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

    model = ResNet18_CBAM(in_channels=1, num_classes=15, feature_dim=256, use_mixstyle=True).to(DEVICE)
    metric_head = SubCenterArcFace(in_features=256, num_classes=15, sub_centers=2).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.08)
    optimizer = torch.optim.AdamW(list(model.parameters()) + list(metric_head.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = GradScaler("cuda") if torch.cuda.is_available() else None

    best_val_acc = 0.0
    best_state = None

    for ep in range(epochs):
        model.train()
        metric_head.train()
        t_loss, correct, total = 0.0, 0, 0

        for imgs, lbls, wrs in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda", enabled=torch.cuda.is_available()):
                logits, feats = model(imgs)
                arc_logits = metric_head(feats, lbls)
                loss = 0.5 * criterion(logits, lbls) + 0.5 * criterion(arc_logits, lbls)

            if scaler:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            t_loss += loss.item() * len(lbls)
            correct += (logits.argmax(dim=1) == lbls).sum().item()
            total += len(lbls)

        scheduler.step()
        train_acc = correct / total

        model.eval()
        metric_head.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, lbls, wrs in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                with autocast("cuda", enabled=torch.cuda.is_available()):
                    logits, feats = model(imgs)
                    arc_logits = metric_head(feats, labels=None)
                p = 0.5 * F.softmax(logits, dim=1) + 0.5 * F.softmax(arc_logits, dim=1)
                val_correct += (p.argmax(dim=1) == lbls).sum().item()
                val_total += len(lbls)

        val_acc = val_correct / val_total
        print(f"    [Base Ep {ep+1:2d}/{epochs:2d}] Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")
        if val_acc >= best_val_acc:
            best_val_acc = val_acc
            best_state = {
                "model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "metric_head_state_dict": {k: v.cpu() for k, v in metric_head.state_dict().items()}
            }

    save_path = os.path.join(MODELS_DIR, base_ckpt_name)
    torch.save(best_state, save_path)
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state["model_state_dict"].items()})
    metric_head.load_state_dict({k: v.to(DEVICE) for k, v in best_state["metric_head_state_dict"].items()})
    print(f"  [+] Base training finished. Best Val Acc: {best_val_acc*100:.2f}%. Saved to {save_path}")
    return model, metric_head




## 8. STAGE 2: SEMI-SUPERVISED CONSISTENCY SELF-TRAINING (FIXMATCH)


In [ ]:
def run_fixmatch_fine_tuning(
    fold: int,
    base_model: nn.Module,
    base_metric_head: nn.Module,
    full_labeled_df: pd.DataFrame,
    val_files: List[str],
    id_to_path: Dict[str, str],
    epochs: int = 4,
    batch_size: int = 64,
    lr: float = 1e-5
) -> Tuple[nn.Module, nn.Module]:
    print("\n" + "=" * 70)
    print(f"⚡ FOLD {fold + 1}/5 | STAGE 2: FIXMATCH CONSISTENCY SELF-TRAINING")
    print(f"   • Labeled Dataset : {len(full_labeled_df)} samples (Base Splits + Synth + Pseudo)")
    print(f"   • Unlabeled Val   : {len(val_files)} samples for Dual-Augmentation Consistency")
    print(f"   • Learning Rate   : {lr} (Controlled Fine-Tuning)")
    print(f"   • Epochs          : {epochs} (Anti-Forgetting Guard)")
    print("=" * 70)

    labeled_ds = SupervisedDataset(full_labeled_df, id_to_path, is_train=True)
    unlabeled_ds = FixMatchDualDataset(val_files)

    labeled_loader = DataLoader(labeled_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available(), drop_last=True)
    unlabeled_loader = DataLoader(unlabeled_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available(), drop_last=True)

    model = base_model
    metric_head = base_metric_head

    criterion_ce = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(list(model.parameters()) + list(metric_head.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    scaler = GradScaler("cuda") if torch.cuda.is_available() else None

    unlabeled_iter = iter(unlabeled_loader)

    for epoch in range(1, epochs + 1):
        model.train()
        metric_head.train()
        epoch_sup_loss = 0.0
        epoch_cons_loss = 0.0
        total_samples = 0

        for imgs_l, lbls_l, _ in labeled_loader:
            try:
                u_w, u_s = next(unlabeled_iter)
            except StopIteration:
                unlabeled_iter = iter(unlabeled_loader)
                u_w, u_s = next(unlabeled_iter)

            imgs_l, lbls_l = imgs_l.to(DEVICE), lbls_l.to(DEVICE)
            u_w, u_s = u_w.to(DEVICE), u_s.to(DEVICE)

            optimizer.zero_grad()

            with autocast("cuda", enabled=torch.cuda.is_available()):
                logits_l, feats_l = model(imgs_l)
                arc_logits_l = metric_head(feats_l, lbls_l)
                loss_sup = 0.5 * criterion_ce(logits_l, lbls_l) + 0.5 * criterion_ce(arc_logits_l, lbls_l)

                with torch.no_grad():
                    logits_u_w, _ = model(u_w)
                    p_u_w = F.softmax(logits_u_w, dim=1)
                    max_p, pseudo_targets = torch.max(p_u_w, dim=1)

                logits_u_s, _ = model(u_s)

                mask = (max_p >= 0.90).float()
                if mask.sum() > 0:
                    ce_per_sample = F.cross_entropy(logits_u_s, pseudo_targets, reduction="none")
                    loss_consistency = (ce_per_sample * mask).sum() / (mask.sum() + 1e-6)
                else:
                    log_p_u_s = F.log_softmax(logits_u_s, dim=1)
                    loss_consistency = -(p_u_w * log_p_u_s).sum(dim=1).mean()

                total_loss = loss_sup + 0.5 * loss_consistency

            if scaler:
                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                optimizer.step()

            B = len(lbls_l)
            epoch_sup_loss += loss_sup.item() * B
            epoch_cons_loss += loss_consistency.item() * B
            total_samples += B

        scheduler.step()
        avg_sup = epoch_sup_loss / total_samples
        avg_cons = epoch_cons_loss / total_samples
        print(f"  [FixMatch Ep {epoch}/{epochs}] Sup Loss: {avg_sup:.4f} | Cons Loss: {avg_cons:.4f} | Joint Loss: {avg_sup + 0.5*avg_cons:.4f}")

    ckpt_save_path = os.path.join(MODELS_DIR, f"resnet18_cbam_fixmatch_fold{fold}.pt")
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "metric_head_state_dict": metric_head.state_dict()
    }, ckpt_save_path)
    print(f"  ⭐ Fold {fold} FixMatch Checkpoint Saved: {ckpt_save_path}")
    return model, metric_head




## 9. STAGE 3: 9-WAY MULTI-SCALE MORPHOLOGICAL TTA EVALUATION


In [ ]:
def get_9way_tta_variants(tensor: torch.Tensor) -> List[torch.Tensor]:
    variants = [tensor]
    B = tensor.size(0)
    dev = tensor.device

    for deg in [3, -3]:
        rad = math.radians(deg)
        theta = torch.tensor([[math.cos(rad), -math.sin(rad), 0],
                              [math.sin(rad),  math.cos(rad), 0]], dtype=torch.float, device=dev).unsqueeze(0).repeat(B, 1, 1)
        grid = F.affine_grid(theta, tensor.size(), align_corners=False)
        variants.append(F.grid_sample(tensor, grid, align_corners=False))

    for dx in [0.045, -0.045]:
        theta = torch.tensor([[1, 0, dx], [0, 1, 0]], dtype=torch.float, device=dev).unsqueeze(0).repeat(B, 1, 1)
        grid = F.affine_grid(theta, tensor.size(), align_corners=False)
        variants.append(F.grid_sample(tensor, grid, align_corners=False))

    theta_x = torch.tensor([[1.05, 0, 0], [0, 1.0, 0]], dtype=torch.float, device=dev).unsqueeze(0).repeat(B, 1, 1)
    grid_x = F.affine_grid(theta_x, tensor.size(), align_corners=False)
    variants.append(F.grid_sample(tensor, grid_x, align_corners=False))

    theta_y = torch.tensor([[1.0, 0, 0], [0, 1.05, 0]], dtype=torch.float, device=dev).unsqueeze(0).repeat(B, 1, 1)
    grid_y = F.affine_grid(theta_y, tensor.size(), align_corners=False)
    variants.append(F.grid_sample(tensor, grid_y, align_corners=False))

    thin = -F.max_pool2d(-tensor, kernel_size=2, stride=1, padding=1)[:, :, :64, :64]
    thick = F.max_pool2d(tensor, kernel_size=2, stride=1, padding=1)[:, :, :64, :64]
    variants.append(thin)
    variants.append(thick)
    return variants


def evaluate_9way_tta_ensemble(
    fine_tuned_models: List[Tuple[nn.Module, nn.Module]],
    val_files: List[str]
) -> Tuple[np.ndarray, pd.DataFrame]:
    print("\n" + "=" * 75)
    print("🔍 RUNNING 9-WAY MULTI-SCALE MORPHOLOGICAL TTA ACROSS ALL 5 FOLDS")
    print(f"   • Target Validation Samples: {len(val_files)}")
    print(f"   • Total Inference Passes   : {len(val_files)} x 9 TTA x 5 Folds = {len(val_files) * 45:,}")
    print("=" * 75)

    all_fold_probs = []

    for fold, (model, metric_head) in enumerate(fine_tuned_models):
        model.eval()
        metric_head.eval()
        fold_probs = []

        with torch.no_grad():
            for fpath in val_files:
                img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    raise IOError(f"Corrupt or missing image: {fpath}")
                if img.shape != (64, 64):
                    img = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)

                t = torch.from_numpy(img.astype(np.float32) / 127.5 - 1.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
                tta_batch = torch.cat(get_9way_tta_variants(t), dim=0)

                with autocast("cuda", enabled=torch.cuda.is_available()):
                    logits, feats = model(tta_batch)
                    arc_logits = metric_head(feats, labels=None)

                p_lin = F.softmax(logits, dim=1)
                p_arc = F.softmax(arc_logits, dim=1)
                p_dual = 0.5 * p_lin + 0.5 * p_arc
                avg_p = p_dual.mean(dim=0).cpu().numpy()
                fold_probs.append(avg_p)

        all_fold_probs.append(np.array(fold_probs))
        print(f"  • Fold {fold + 1}/5 9-Way TTA Evaluation Completed.")

    ensemble_probs = np.mean(all_fold_probs, axis=0)
    val_ids = [os.path.splitext(os.path.basename(f))[0] for f in val_files]
    pred_labels = np.argmax(ensemble_probs, axis=1)
    df_sub = pd.DataFrame({"id": val_ids, "label": pred_labels})
    return ensemble_probs, df_sub




## 10. MAIN EXECUTION PIPELINE


In [ ]:
def main():
    splits_csv, synth_trad_csv, synth_callig_csv, id_to_path, val_files, pseudo_map = setup_datasets()
    df_splits = pd.read_csv(splits_csv)

    df_synth_trad = pd.read_csv(synth_trad_csv) if synth_trad_csv else pd.DataFrame()
    df_synth_callig = pd.read_csv(synth_callig_csv) if synth_callig_csv else pd.DataFrame()

    pseudo_records = []
    for val_id, plabel in pseudo_map.items():
        if val_id in id_to_path or f"{val_id}.png" in id_to_path or f"{val_id}.jpg" in id_to_path:
            pseudo_records.append({"id": val_id, "label": plabel, "writer_id": 888})

    df_pseudo = pd.DataFrame(pseudo_records)
    print("\n[*] FixMatch Training Corpus Composition:")
    print(f"    • Base Splits Data      : {len(df_splits)} samples")
    print(f"    • Synthetic Traditional : {len(df_synth_trad)} samples")
    print(f"    • Synthetic Calligraphy : {len(df_synth_callig)} samples")
    print(f"    • Consensus Pseudo-Data : {len(df_pseudo)} samples (>98% confidence)")

    fine_tuned_models = []

    for fold in range(5):
        print("\n" + "#" * 75)
        print(f"### EXECUTING FIXMATCH PIPELINE — FOLD {fold + 1} / 5 ###")
        print("#" * 75)

        train_split = df_splits[df_splits["fold"] != fold].copy()
        val_split = df_splits[df_splits["fold"] == fold].copy()

        train_dfs = [train_split]
        if not df_synth_trad.empty:
            train_dfs.append(df_synth_trad)
        if not df_synth_callig.empty:
            train_dfs.append(df_synth_callig)

        base_model, base_metric_head = get_or_train_base_model(
            fold=fold,
            train_df=pd.concat(train_dfs, ignore_index=True),
            val_df=val_split,
            id_to_path=id_to_path,
            epochs=10,
            batch_size=64,
            lr=5e-4
        )

        fixmatch_labeled_df = pd.concat(train_dfs + [df_pseudo], ignore_index=True)

        ft_model, ft_head = run_fixmatch_fine_tuning(
            fold=fold,
            base_model=base_model,
            base_metric_head=base_metric_head,
            full_labeled_df=fixmatch_labeled_df,
            val_files=val_files,
            id_to_path=id_to_path,
            epochs=4,
            batch_size=64,
            lr=1e-5
        )

        fine_tuned_models.append((ft_model, ft_head))

    ensemble_probs, df_sub = evaluate_9way_tta_ensemble(fine_tuned_models, val_files)

    prob_save_path = os.path.join(WORKING_DIR, "probabilities_fixmatch_5fold.npy")
    np.save(prob_save_path, ensemble_probs)
    print(f"\n✅ [1/3] Saved Probabilities to: {prob_save_path}")

    csv_save_path = os.path.join(WORKING_DIR, "submission_fixmatch.csv")
    df_sub.to_csv(csv_save_path, index=False)
    print(f"✅ [2/3] Saved Submission CSV to: {csv_save_path}")

    zip_save_path = os.path.join(WORKING_DIR, "sub_fixmatch.zip")
    with zipfile.ZipFile(zip_save_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_save_path, arcname="predictions.csv")
    print(f"✅ [3/3] Created Submission Zip Archive: {zip_save_path} ({os.path.getsize(zip_save_path)} bytes)")

    print("\n" + "=" * 75)
    print("🎯 FIXMATCH TRACK 2 PIPELINE EXECUTION COMPLETED SUCCESSFULLY!")
    print(f"Total Samples: {len(df_sub)}")
    for c in range(15):
        print(f"  Class {c:2d}: {(df_sub['label'] == c).sum()} samples")
    print("=" * 75)


if __name__ == "__main__":
    main()
